** Rules **

- R1. Three-way split: test = 20% of all rows (random_state=42, stratify), then validation = 25% of the remaining rows (random_state=42, stratify). So about 60/20/20.

- R2. The median fill and the StandardScaler are fitted on the inner TRAIN rows only.

- R3. The model returns raw logits (no Sigmoid inside it). Loss = nn.BCEWithLogitsLoss().
Optimizer = Adam, lr = 0.01. Full batch (all rows at once) for 300 epochs.

- R4. Choose on validation AUC. The hidden-layer network must beat the single neuron by MORE
than 0.005 AUC, otherwise choose the simpler model.

R5. The test set is opened ONCE, for the chosen model only.


In [1]:
import torch, torch.nn as nn
print(torch.__version__)

2.11.0+cu130


In [2]:
import math, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

In [19]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["churn"] = (df["Churn"] == "Yes").astype(int)

num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = ["Contract", "InternetService", "PaymentMethod",
        "PaperlessBilling", "Partner", "Dependents", "TechSupport"]

X = pd.get_dummies(df[num + cat], columns=cat, dtype=int)
y = df["churn"]

X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_tr, X_val, y_tr, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)

med = X_tr["TotalCharges"].median()
for d in (X_tr, X_val, X_test):
    d["TotalCharges"] = d["TotalCharges"].fillna(med)

sc = StandardScaler().fit(X_tr)

Xtr = torch.tensor(sc.transform(X_tr), dtype=torch.float32)
Xval = torch.tensor(sc.transform(X_val), dtype=torch.float32)
Xtest = torch.tensor(sc.transform(X_test), dtype=torch.float32)

ytr = torch.tensor(y_tr.values, dtype=torch.float32).unsqueeze(1)
yval = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
ytest = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

print(med)
print(X_tr.isna().sum().sum(), X_val.isna().sum().sum(), X_test.isna().sum().sum())
print(Xtr.shape, ytr.shape, round(ytr.mean().item(), 4), round(yval.mean().item(), 4))
print(Xtest.shape, Xval.shape)
print("churners: train", int(y_tr.sum()), "| validation", int(y_val.sum()), "| test", int(y_test.sum()))

1385.3
0 0 0
torch.Size([4225, 23]) torch.Size([4225, 1]) 0.2653 0.2654
torch.Size([1409, 23]) torch.Size([1409, 23])
churners: train 1121 | validation 374 | test 374


Write one sentence: why is the median different from the Day 13 one (1398.125)?
- In Day 13 the training set was all 5,634 rows. This time 1,409 of them were moved into validation, so the remaining training rows are different and have a slightly different median.

In [29]:
loss_fn = nn.BCEWithLogitsLoss()

from sklearn.metrics import roc_auc_score, recall_score, precision_score

def fit(model, epochs, lr=0.01, show=()):
  opt = torch.optim.Adam(model.parameters(), lr=lr)
  for e in range(1, epochs+1):
    model.train()
    opt.zero_grad()
    l = loss_fn(model(Xtr), ytr)
    l.backward()
    opt.step()
    if e in show:
      model.eval()
      with torch.no_grad():
        lv = loss_fn(model(Xval), yval).item()
        pv = torch.sigmoid(model(Xval))
      print(e, round(l.item(), 4), round(lv, 4), round(roc_auc_score(yval.numpy(), pv.numpy()), 4))
  return model

def evalm(model, Xs, ys, t=0.5):
  model.eval()
  with torch.no_grad():
    p = torch.sigmoid(model(Xs)).numpy().ravel()
  yy = ys.numpy().ravel()
  pr = (p >= t).astype(int)
  return round(roc_auc_score(yy, p), 4), round(recall_score(yy, pr), 4), round(precision_score(yy, pr), 4)

Write one sentence: model.eval() changes nothing for our models. Why is it still a good habit?
(Hint: layers such as Dropout and BatchNorm, which you meet on Day 17, behave differently.)
- It is good habit because it will change how the model behaves when we perform Dropout or BatchNorm.

In [31]:
#Model A: One Neuron (Logistic regression in Pytorch)
torch.manual_seed(42)
A = nn.Linear(23, 1)
fit(A, 300, show=(1, 50, 100, 200, 300))
print(sum(p.numel() for p in A.parameters()), "parameters")

1 0.7295 0.7158 0.6341
50 0.5118 0.5175 0.8281
100 0.4598 0.4661 0.8306
200 0.4275 0.4367 0.8317
300 0.4206 0.4317 0.8325
24 parameters


In [37]:
from sklearn.linear_model import LogisticRegression

lg = LogisticRegression(max_iter=1000).fit(sc.transform(X_tr), y_tr)
print(round(roc_auc_score(y_val, lg.predict_proba(sc.transform(X_val))[:, 1]), 4))

0.8357


Write two sentences:
- Count the 24 parameters by hand.
  * (23 * 1 + 1) = 24
- A single Linear(23, 1) followed by a sigmoid IS the Day 9 logistic regression. Why is its
AUC (0.8325) a little lower than sklearn's (0.8357)? (Think: how many epochs, which
optimizer.)
  * sklearn’s solver runs until it converges. Adam at lr = 0.01 for 300 steps has not fully converged, so if it does not have enough epoch to run, it will stop before reaching the perfect point.


In [38]:
#PART 4. MODEL B: ONE HIDDEN LAYER
torch.manual_seed(42)
B = nn.Sequential(nn.Linear(23, 16), nn.ReLU(), nn.Linear(16, 1))
fit(B, 300, show=(1, 50, 100, 200, 300))
print(sum(p.numel() for p in B.parameters()), "parameters")

1 0.6619 0.634 0.7412
50 0.4135 0.4362 0.8288
100 0.4037 0.4352 0.8324
200 0.3906 0.4459 0.8238
300 0.3866 0.4501 0.8212
401 parameters


Count the 401 by hand: (23 x 16 + 16) + (16 x 1 + 1) = 401
Read the table carefully, then write three sentences:

- What happens to the TRAIN loss from epoch 100 to 300?
  * gradually decrease from 0.4037 -> 0.3886 as the epoch increases.

- What happens to the VALIDATION loss and validation AUC over the same epochs?
  * the validation loss increases from 0.4352 -> 0.4501 and validation auc decrease from 0.8324 -> 0.8212

- Which Day 11 word describes this, and which epoch would you have stopped at?
  * Overfitting because the model is starting to memorize training data (train loss decreases, val loss keep increasing) and i would stop at epoch 100 because at epoch 100 val loss reach the lowest point (0.4352) and val auc hit the peak (0.8324)

In [43]:
#PART 5. MODEL C: A BIGGER NETWORK
torch.manual_seed(42)
C = nn.Sequential(nn.Linear(23, 64), nn.ReLU(), nn.Linear(64, 1))
fit(C, 1500, show=(100, 300, 500, 1000, 1500))
print(sum(p.numel() for p in C.parameters()), "parameters")

100 0.3902 0.4414 0.8263
300 0.3389 0.4907 0.8032
500 0.3194 0.5381 0.7889
1000 0.3013 0.6171 0.7708
1500 0.2863 0.6695 0.7764
1601 parameters


Write two sentences:
- The train loss keeps falling while the validation AUC falls from 0.8263 to 0.7741. What is
the network doing to the training rows?
  * network is overfitting by memorising the specific training rows instead of learning general patterns.

- Does more capacity (64 hidden neurons, 1500 epochs) help on this data? Link it to the
rows-per-feature idea from Day 11.
  * No, because increasing capacity and epochs gives the model too much freedom to memorize the data. Because the rows-per-feature ratio becomes too low, the model overfits instead of learning general patterns.


In [44]:
for n, m in (("A", A), ("B", B), ("C", C)):
  print(n, "val", evalm(m, Xval, yval), "| at 0.35", evalm(m, Xval, yval, 0.35))

A val (np.float64(0.8325), 0.5481, 0.6571) | at 0.35 (np.float64(0.8325), 0.7299, 0.5652)
B val (np.float64(0.8212), 0.5187, 0.634) | at 0.35 (np.float64(0.8212), 0.6711, 0.5433)
C val (np.float64(0.7764), 0.5535, 0.5308) | at 0.35 (np.float64(0.7764), 0.6604, 0.501)


Apply R4 in writing: does B beat A by more than 0.005? Which model do you choose and why?

In [45]:
print("test", evalm(A, Xtest, ytest), "| at 0.35", evalm(A, Xtest, ytest, 0.35))

test (np.float64(0.832), 0.5428, 0.6133) | at 0.35 (np.float64(0.832), 0.7193, 0.5264)


Write the comparison with Day 12: logistic regression 0.8383, tuned forest 0.8410 (test AUC).
  * Network scored 0.832, logistic regression 0.8383 and the tuned forest 0.8410. The gaps are 0.0063 and 0.0090.
- Your network scored 0.832. Give two reasons it is a little lower than 0.8383. (Hints: it
was trained on 4225 rows, not 5634; Adam for 300 epochs, not a fully converged solver.)
  * neural network was trained on 4,225 rows because a chunk of the data had to be held out for the validation set to monitor overfitting. On the other hand, the scikit-learn Logistic Regression model had access to all 5,634 rows at once. Network used the Adam optimizer for 300 epochs, which updates weights using noisy, Adam at lr = 0.01 for 300 steps has not fully converged.

- Would you replace the Day 12 logistic regression with this network? One sentence.
Do NOT also score B or C on the test set. R5 says once, for the chosen model.
  * No, because the Day 12 version had more training rows and a converged solver.


** Conclusion **
- A neuron is a single processing unit that calculates a weighted sum of its inputs plus a bias; a layer is a collection of these neurons working together in parallel; and an activation function (like ReLU) introduces non-linearity, allowing the network to learn complex, non-linear relationships rather than just simple straight lines.

- A training loop relies on clearing past gradients (.zero_grad), running a forward pass to get predictions (model(inputs)), calculating error, • computing new gradients via backpropagation (.backward()), and updating the model's weights (.step()).

- Using BCEWithLogitsLoss without a Sigmoid layer inside the model is preferred because it mathematically combines the Sigmoid operation and the Binary Cross-Entropy loss into a single step.

- Model A (24 parameters) represents a low-capacity model that scales well over 300 epochs because its validation loss keeps falling (0.7158 → 0.4317) while its validation AUC climbs steadily to its peak of 0.8325, showing no signs of overfitting.
- Model B (401 parameters) hits its highest performance at epoch 100 (0.4352, 0.8324) but begins to overfit afterward as its training loss continues to drop while its validation loss increasing.
- Model C• Model C (1,601 parameters) suffers from severe overcapacity, causing it to hard-memorize the training rows; its training loss plummets down to 0.2863 while its validation loss balloons to 0.6695 and its validation AUC crashes from 0.8263 down to 0.7764 by epoch 1500.

- To counter this overfitting and improve generalization on Day 17, I would implement dropout to randomly deactivate neurons and prevent co-dependency, add weight decay (L2 regularization) to penalize excessively large weights, and use early stopping to automatically freeze the model at its absolute peak validation performance.